# Notebook 03 - Synthetic Data Generation

**Topic:** Data Collection Strategies  
**Task:** Use `Faker` and `numpy` to generate a fake e-commerce orders dataset with 1000 rows

---

## What you will learn

- How to generate realistic fake data using the `Faker` library
- How to model real-world statistical distributions with `numpy`
- How to introduce correlations between variables to make synthetic data believable
- How to validate that synthetic data matches expected statistical properties
- How to build a reusable data generator function

---

## Prerequisites

```bash
pip install faker pandas numpy matplotlib
```

---

## Section 1 - Imports and Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os

from faker import Faker
from datetime import datetime, timedelta

SEED = 42
np.random.seed(SEED)
fake = Faker()
Faker.seed(SEED)

os.makedirs("output", exist_ok=True)

plt.rcParams["figure.figsize"] = (11, 4)
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

print("Libraries ready. Random seed set to", SEED)

---

## Section 2 - Why Synthetic Data?

Synthetic data is generated programmatically rather than collected from the world. It is used when:

1. **Real data is private** - medical records, financial transactions, personal identifiers
2. **Real data is scarce** - rare events like equipment failures or fraud cases
3. **You need labeled data** - generating ground truth for ML training
4. **You are building a pipeline** - you need rows to test code before real data arrives

The goal is not perfect realism. The goal is that the statistical properties of your synthetic data are close enough to real data that models trained or tested on it behave predictably.

---

## Section 3 - Exploring the Faker Library

`Faker` is a Python library that generates realistic fake text data: names, addresses, emails, dates, company names, and much more.

In [ ]:
# Preview what Faker can generate
print("Name         :", fake.name())
print("Email        :", fake.email())
print("Phone        :", fake.phone_number())
print("City         :", fake.city())
print("Country      :", fake.country())
print("Company      :", fake.company())
print("UUID         :", fake.uuid4())
print("Date (3y ago):", fake.date_between(start_date="-3y", end_date="today"))
print("Credit card  :", fake.credit_card_number(card_type="visa"))
print("User agent   :", fake.user_agent())

---

## Section 4 - Designing the E-Commerce Schema

Before writing any code, we define what the dataset should look like. A real e-commerce orders table might include:

| Column | Type | Notes |
|---|---|---|
| `order_id` | string | Unique identifier |
| `customer_id` | string | Unique per customer |
| `customer_name` | string | Full name |
| `email` | string | Customer email |
| `country` | string | Shipping country |
| `order_date` | date | Within the last 2 years |
| `product_category` | string | One of 5 categories |
| `product_name` | string | Specific product |
| `quantity` | int | 1 to 10 |
| `unit_price_usd` | float | Varies by category |
| `total_price_usd` | float | quantity x unit_price |
| `discount_pct` | float | 0 to 40% discount |
| `final_price_usd` | float | After discount |
| `payment_method` | string | Card, PayPal, etc. |
| `order_status` | string | Delivered, Pending, Cancelled |
| `is_returned` | bool | Whether the order was returned |

**Key design decisions:**
- `unit_price_usd` should vary by product category (electronics cost more than books)
- `is_returned` should be more likely for expensive orders (a realistic correlation)
- `order_status` should be biased: most orders get delivered
- Customers can have multiple orders (repeat customers)

---

## Section 5 - Building the Generator

### 5.1 Define category-level parameters

We use dictionaries to store the configuration for each product category. This makes the code easy to extend.

In [ ]:
# Each category has a list of products and a price distribution (mean, std)
CATEGORIES = {
    "Electronics": {
        "products" : ["Wireless Headphones", "USB-C Hub", "Mechanical Keyboard",
                      "Webcam HD", "Portable SSD", "Smart Watch", "Phone Stand"],
        "price_mean": 120.0,
        "price_std" : 60.0,
        "price_min" : 15.0,
        "price_max" : 500.0,
    },
    "Books": {
        "products" : ["Python for Data Science", "Clean Code", "Deep Learning",
                      "The Pragmatic Programmer", "Statistics for ML", "SQL Cookbook"],
        "price_mean": 28.0,
        "price_std" : 10.0,
        "price_min" : 8.0,
        "price_max" : 80.0,
    },
    "Clothing": {
        "products" : ["Running Shoes", "Denim Jacket", "Polo Shirt",
                      "Yoga Pants", "Hoodie", "Cotton Socks"],
        "price_mean": 55.0,
        "price_std" : 30.0,
        "price_min" : 5.0,
        "price_max" : 250.0,
    },
    "Home & Kitchen": {
        "products" : ["Coffee Maker", "Non-Stick Pan", "Air Purifier",
                      "Knife Set", "Blender", "Electric Kettle"],
        "price_mean": 75.0,
        "price_std" : 45.0,
        "price_min" : 10.0,
        "price_max" : 400.0,
    },
    "Sports": {
        "products" : ["Resistance Bands", "Yoga Mat", "Dumbbells",
                      "Water Bottle", "Fitness Tracker", "Jump Rope"],
        "price_mean": 45.0,
        "price_std" : 25.0,
        "price_min" : 5.0,
        "price_max" : 200.0,
    },
}

CATEGORY_NAMES = list(CATEGORIES.keys())

# Category selection probabilities: Electronics is purchased most frequently
CATEGORY_WEIGHTS = [0.30, 0.20, 0.20, 0.15, 0.15]

PAYMENT_METHODS = ["Credit Card", "Debit Card", "PayPal", "Bank Transfer", "Crypto"]
PAYMENT_WEIGHTS = [0.40, 0.25, 0.20, 0.10, 0.05]

ORDER_STATUSES = ["Delivered", "Pending", "Shipped", "Cancelled"]
STATUS_WEIGHTS  = [0.72, 0.10, 0.12, 0.06]

# Countries weighted by rough e-commerce volume
COUNTRIES = ["United States", "United Kingdom", "Germany", "France", "Canada",
             "Australia", "India", "Brazil", "Japan", "Netherlands"]
COUNTRY_WEIGHTS = [0.30, 0.15, 0.10, 0.08, 0.08, 0.07, 0.07, 0.05, 0.05, 0.05]

print("Configuration ready.")
print(f"Categories : {CATEGORY_NAMES}")
print(f"Countries  : {len(COUNTRIES)}")

### 5.2 Build a pool of repeat customers

Real e-commerce data has repeat customers. We generate a pool of customers first, then assign orders to them.

In [ ]:
def generate_customer_pool(n_customers=300):
    """
    Generate a pool of unique customers.
    Returns a list of dicts, each with a customer_id, name, email, and country.
    """
    customers = []
    for _ in range(n_customers):
        name = fake.name()
        # Derive email from name to make it look realistic
        first, *_, last = name.lower().split()
        domain = fake.free_email_domain()
        email = f"{first}.{last}{np.random.randint(1, 99)}@{domain}"

        customers.append({
            "customer_id"  : fake.uuid4()[:12].upper(),
            "customer_name": name,
            "email"        : email,
            "country"      : np.random.choice(COUNTRIES, p=COUNTRY_WEIGHTS),
        })

    return customers


customer_pool = generate_customer_pool(300)
print(f"Generated {len(customer_pool)} customers.")
print("Sample customer:", customer_pool[0])

### 5.3 Generate a single order row

In [ ]:
def generate_order(customer):
    """
    Generate a single order record for a given customer.

    Design choices that reflect real-world patterns:
    - Price follows a truncated normal distribution per category.
    - Discount probability is higher for expensive items.
    - Return probability is positively correlated with final price.
    - Order date uses a recency bias (more recent orders are more common).
    """
    # Pick category and product
    category_name = np.random.choice(CATEGORY_NAMES, p=CATEGORY_WEIGHTS)
    cat = CATEGORIES[category_name]
    product = np.random.choice(cat["products"])

    # Generate price from a normal distribution clipped to realistic bounds
    raw_price = np.random.normal(cat["price_mean"], cat["price_std"])
    unit_price = float(np.clip(raw_price, cat["price_min"], cat["price_max"]))
    unit_price = round(unit_price, 2)

    quantity = int(np.random.choice([1, 2, 3, 4, 5], p=[0.55, 0.25, 0.10, 0.06, 0.04]))
    total_price = round(unit_price * quantity, 2)

    # Higher-value items are more likely to have a discount
    discount_prob = min(0.5, total_price / 1000)
    has_discount = np.random.rand() < discount_prob
    discount_pct = round(np.random.uniform(5, 40), 1) if has_discount else 0.0
    final_price = round(total_price * (1 - discount_pct / 100), 2)

    # Order date with recency bias (exponential distribution, more recent dates more common)
    days_ago = int(np.random.exponential(scale=180))
    days_ago = min(days_ago, 730)  # Cap at 2 years
    order_date = datetime.today().date() - timedelta(days=days_ago)

    # Return probability scales with price (expensive orders returned more)
    return_prob = min(0.30, final_price / 2000)
    is_returned = bool(np.random.rand() < return_prob)

    status = np.random.choice(ORDER_STATUSES, p=STATUS_WEIGHTS)
    # A returned order cannot also be cancelled or pending
    if is_returned:
        status = "Delivered"

    return {
        "order_id"          : "ORD-" + fake.uuid4()[:8].upper(),
        "customer_id"       : customer["customer_id"],
        "customer_name"     : customer["customer_name"],
        "email"             : customer["email"],
        "country"           : customer["country"],
        "order_date"        : order_date,
        "product_category"  : category_name,
        "product_name"      : product,
        "quantity"          : quantity,
        "unit_price_usd"    : unit_price,
        "total_price_usd"   : total_price,
        "discount_pct"      : discount_pct,
        "final_price_usd"   : final_price,
        "payment_method"    : np.random.choice(PAYMENT_METHODS, p=PAYMENT_WEIGHTS),
        "order_status"      : status,
        "is_returned"       : is_returned,
    }


# Test a single row
sample_order = generate_order(customer_pool[0])
for k, v in sample_order.items():
    print(f"  {k:<22}: {v}")

### 5.4 Generate the full dataset

In [ ]:
def generate_orders_dataset(n_orders=1000, customer_pool=None):
    """
    Generate a DataFrame of n_orders e-commerce order records.

    Parameters
    ----------
    n_orders      : int  - Number of rows to generate
    customer_pool : list - Pre-generated list of customer dicts

    Returns
    -------
    pd.DataFrame
    """
    if customer_pool is None:
        customer_pool = generate_customer_pool(n_orders // 3)

    orders = []
    for _ in range(n_orders):
        customer = np.random.choice(customer_pool)  # Allows repeat customers
        orders.append(generate_order(customer))

    df = pd.DataFrame(orders)
    df["order_date"] = pd.to_datetime(df["order_date"])
    df = df.sort_values("order_date").reset_index(drop=True)

    return df


df_orders = generate_orders_dataset(n_orders=1000, customer_pool=customer_pool)

print(f"Generated dataset shape: {df_orders.shape}")
df_orders.head()

---

## Section 6 - Validating the Synthetic Data

After generating synthetic data, you must verify that it actually has the statistical properties you intended. This step is often skipped but is critical.

In [ ]:
print("[1] Basic shape and types")
print(f"    Rows: {df_orders.shape[0]}, Columns: {df_orders.shape[1]}")
print(f"    Missing values: {df_orders.isnull().sum().sum()}")
print(f"    Duplicate order IDs: {df_orders['order_id'].duplicated().sum()}")

In [ ]:
print("[2] Category distribution (expected: Electronics ~30%, Books ~20%)")
print(df_orders["product_category"].value_counts(normalize=True).round(3).to_string())

In [ ]:
print("[3] Price statistics by category")
print(
    df_orders.groupby("product_category")["unit_price_usd"]
    .describe()[["mean", "std", "min", "max"]]
    .round(2)
    .to_string()
)

In [ ]:
print("[4] Return rate by category (expensive categories should have higher returns)")
return_by_cat = df_orders.groupby("product_category")["is_returned"].mean().sort_values(ascending=False)
print(return_by_cat.round(3).to_string())

In [ ]:
print("[5] Repeat customers (some customer IDs should appear more than once)")
orders_per_customer = df_orders["customer_id"].value_counts()
print(f"    Unique customers    : {df_orders['customer_id'].nunique()}")
print(f"    Max orders (1 cust) : {orders_per_customer.max()}")
print(f"    Avg orders/customer : {orders_per_customer.mean():.2f}")
print(f"    Customers with 3+   : {(orders_per_customer >= 3).sum()}")

In [ ]:
print("[6] Verify correlation: final_price and is_returned should be positively correlated")
corr = df_orders[["final_price_usd", "is_returned"]].corr().iloc[0, 1]
print(f"    Pearson correlation (final_price vs is_returned): {corr:.3f}")
print("    Expected: positive value (more expensive = more returns)")

---

## Section 7 - Visualising the Synthetic Data

In [ ]:
fig, axes = plt.subplots(1, 3)

# Plot 1: Orders by category
cat_counts = df_orders["product_category"].value_counts()
cat_counts.plot(kind="barh", ax=axes[0], color="#5b8dd9")
axes[0].set_title("Orders by category")
axes[0].set_xlabel("Count")

# Plot 2: Final price distribution
df_orders["final_price_usd"].plot(kind="hist", bins=40, ax=axes[1], color="#52b788")
axes[1].set_title("Final price distribution")
axes[1].set_xlabel("Final price (USD)")

# Plot 3: Order volume over time (monthly)
monthly = df_orders.set_index("order_date").resample("ME")["order_id"].count()
monthly.plot(ax=axes[2], color="#e86d6d", linewidth=1.8)
axes[2].set_title("Monthly order volume")
axes[2].set_ylabel("Orders")

plt.tight_layout()
plt.savefig("output/synthetic_orders_overview.png", dpi=120)
plt.show()

In [ ]:
# Return rate vs average price bucket
df_orders["price_bucket"] = pd.cut(
    df_orders["final_price_usd"],
    bins=[0, 25, 50, 100, 200, 500, 2000],
    labels=["0-25", "25-50", "50-100", "100-200", "200-500", "500+"]
)

return_by_bucket = df_orders.groupby("price_bucket", observed=True)["is_returned"].mean()

fig, ax = plt.subplots(figsize=(8, 4))
return_by_bucket.plot(kind="bar", ax=ax, color="#e86d6d", width=0.55)
ax.set_title("Return rate by price bucket (expected to increase with price)")
ax.set_xlabel("Final price range (USD)")
ax.set_ylabel("Return rate")
ax.set_xticklabels(return_by_bucket.index, rotation=0)
plt.tight_layout()
plt.savefig("output/synthetic_return_rate.png", dpi=120)
plt.show()

print("Interpretation: Return rate rises with order value, as designed.")

---

## Section 8 - Save the Dataset

In [ ]:
# Drop the helper column before saving
df_final = df_orders.drop(columns=["price_bucket"])

df_final.to_csv("output/ecommerce_orders_synthetic.csv", index=False)
print(f"Saved ecommerce_orders_synthetic.csv with {len(df_final)} rows and {df_final.shape[1]} columns.")
print("\nColumn list:")
print(df_final.columns.tolist())

---

## Section 9 - Generating Larger Datasets Efficiently

The row-by-row generator above is clear to read but slow at scale. For 100k+ rows, a vectorised approach using `numpy` is much faster.

In [ ]:
def generate_orders_vectorised(n=100_000):
    """
    Fast vectorised generator for large e-commerce datasets.
    Trades some flexibility for speed by using numpy arrays throughout.
    """
    cat_idx = np.random.choice(len(CATEGORY_NAMES), size=n, p=CATEGORY_WEIGHTS)
    categories = [CATEGORY_NAMES[i] for i in cat_idx]

    price_means = np.array([CATEGORIES[c]["price_mean"] for c in categories])
    price_stds  = np.array([CATEGORIES[c]["price_std"]  for c in categories])
    price_mins  = np.array([CATEGORIES[c]["price_min"]  for c in categories])
    price_maxs  = np.array([CATEGORIES[c]["price_max"]  for c in categories])

    raw_prices  = np.random.normal(price_means, price_stds)
    unit_prices = np.clip(raw_prices, price_mins, price_maxs).round(2)

    quantities   = np.random.choice([1, 2, 3, 4, 5], size=n, p=[0.55, 0.25, 0.10, 0.06, 0.04])
    total_prices = (unit_prices * quantities).round(2)

    discount_probs = np.minimum(0.5, total_prices / 1000)
    has_discount   = np.random.rand(n) < discount_probs
    discount_pcts  = np.where(has_discount, np.random.uniform(5, 40, n).round(1), 0.0)
    final_prices   = (total_prices * (1 - discount_pcts / 100)).round(2)

    return_probs = np.minimum(0.30, final_prices / 2000)
    is_returned  = np.random.rand(n) < return_probs

    return pd.DataFrame({
        "product_category" : categories,
        "quantity"         : quantities,
        "unit_price_usd"   : unit_prices,
        "total_price_usd"  : total_prices,
        "discount_pct"     : discount_pcts,
        "final_price_usd"  : final_prices,
        "is_returned"      : is_returned,
    })


import time
start = time.time()
df_large = generate_orders_vectorised(100_000)
elapsed = time.time() - start

print(f"Generated 100,000 rows in {elapsed:.2f} seconds.")
print(df_large.shape)

---

## Section 10 - Summary

| Concept | What you did |
|---|---|
| Faker basics | Generated names, emails, UUIDs, and dates |
| Customer pool | Created 300 unique customers to allow repeat orders |
| Distribution design | Used truncated normal distributions per category for prices |
| Correlation design | Made return rate depend on final price |
| Validation | Checked distribution, correlations, and repeat customer behaviour |
| Vectorised generation | Used numpy arrays for 100k rows in under 1 second |

**Files saved in the `output/` folder:**
- `ecommerce_orders_synthetic.csv` - 1000 rows, 16 columns
- `synthetic_orders_overview.png`
- `synthetic_return_rate.png`

---

**Practice challenge:** Add a new column `shipping_days` that depends on `country` (domestic orders ship in 1-3 days, international in 5-14 days) and on `order_status` (cancelled orders have no shipping days). Update the generator and re-run the validation section.